# Model Golf
**MassAI · November 18, 2026**

How few bytes can your heartbeat classifier use while still catching PVCs? Work in teams of 2–3. Have one person run trials and another compare the results. Set aside 17 minutes for the four stages after setup.

Open this notebook in [Colab](https://colab.research.google.com/) via **File → Upload notebook**. Use the CPU runtime. Run setup and choose `model-golf-data.zip` from the student kit when prompted. For local Jupyter, keep that ZIP beside the notebook.

In [ ]:
from pathlib import Path
from zipfile import ZipFile
import sys

archive_name = 'model-golf-data.zip'
base = Path('.')
if not (base / 'classroom.npz').exists():
    if not (base / archive_name).exists():
        try:
            from google.colab import files
        except ImportError as error:
            raise FileNotFoundError(f'Put {archive_name} beside this notebook.') from error
        print(f'Choose {archive_name} in the upload dialog.')
        uploaded = files.upload()
        if archive_name not in uploaded:
            raise ValueError(f'Please upload the supplied {archive_name}.')
        (base / archive_name).write_bytes(uploaded[archive_name])
    base = Path('model-golf-data')
    members = ('classroom.npz', 'baseline.mgf', 'model_golf.py', 'rules.json')
    with ZipFile(archive_name) as archive:
        for name in members:
            target = base / name
            target.parent.mkdir(parents=True, exist_ok=True)
            target.write_bytes(archive.read(name))

import numpy as np
sys.path.insert(0, str(base.resolve()))
from model_golf import (BASELINE_WIDTH, INPUTS, SEED, TRAINING_EPOCHS, TRAINING_SEEDS,
                        WIDTHS, decode, encode, format_score, score, train)

with np.load(base / 'classroom.npz', allow_pickle=False) as data:
    x_train, y_train = data['train_x'], data['train_y']
    x_val, y_val = data['validation_x'], data['validation_y']
baseline = (base / 'baseline.mgf').read_bytes()
print(f'{len(x_train):,} training windows · {len(x_val):,} practice windows · {x_train.shape[1]} values per window')

## A · Baseline — 2 minutes

Each input is a window around an annotated heartbeat from MIT-BIH. The model distinguishes **N** (normal beat) from **V** (premature ventricular contraction, or PVC). It reads 64 ECG values, combines them in one hidden layer, and outputs a score. A score of zero or higher predicts V.

**Accuracy** is the share of all beats classified correctly. **PVC sensitivity** is the share of annotated PVC beats the model catches. Normal beats outnumber PVCs, so guessing N every time can score well on accuracy while missing every PVC.

In [ ]:
import json

rules = json.loads((base / 'rules.json').read_text())
accuracy_floor = rules['minimum_accuracy']
sensitivity_floor = rules['minimum_pvc_sensitivity']
baseline_model = decode(baseline)
parameters = baseline_model.w1.size + baseline_model.b1.size + baseline_model.w2.size + 1
baseline_score = score(baseline, x_val, y_val)

print(f'{INPUTS} ECG values → {BASELINE_WIDTH} ReLU neurons → 1 score')
print(f'{parameters:,} learned parameters · FP32 weights')
print(f"Practice: {baseline_score['bytes']:,} bytes · {baseline_score['accuracy']:.2%} accuracy · "
      f"{baseline_score['sensitivity']:.2%} PVC sensitivity")
print(f"Correct N: {baseline_score['tn']:,}  |  N flagged as PVC: {baseline_score['fp']:,}")
print(f"Missed PVC: {baseline_score['fn']:,}  |  Correct PVC: {baseline_score['tp']:,}")
print(f'Always predict N: {np.mean(y_val == 0):.1%} accuracy, 0% PVC sensitivity')
print(f'Final scoring floors: {accuracy_floor:.0%} accuracy and {sensitivity_floor:.0%} PVC sensitivity')

## B · First optimization — 4 minutes

Start with the same trained model. A 32-bit floating-point weight takes four bytes. INT8 stores a rounded weight in one byte and adds a scale for each layer. The simulator uses those scales to recover approximate weights and still accumulates in FP32. The model keeps the same number of neurons and parameters.

Before running the cell, guess how many bytes you’ll save. Compare the size and both practice metrics after the conversion. This exercise uses pretrained weights and runs without training.

In [ ]:
int8_baseline = encode(baseline_model, bits=8)
int8_score = score(int8_baseline, x_val, y_val)
saved = len(baseline) - len(int8_baseline)

print('FP32 before:', format_score(baseline_score))
print('INT8 after: ', format_score(int8_score))
print(f'Saved {saved:,} bytes ({saved / len(baseline):.1%}) with {parameters:,} parameters in both models.')

## C · Model Golf — 10 minutes

Edit the trial settings and rerun that cell. Try **16**, **8**, then **4** hidden neurons with INT8. Fewer neurons mean fewer weights and biases. Change `bits` to **32** on one of your widths to isolate what quantization changed.

| Setting | Choices | What it changes |
|---|---|---|
| `width` | `4`, `8`, `16`, `24` | Number of neurons in the hidden layer |
| `bits` | `8`, `32` | Precision of stored weights |
| `seed` | `18`, `7`, `42` | Initial weights, sampled normal beats, and training order |
| `training` | `"standard"`, `"short"` | 45 or 15 passes through the training subset |

Leave `seed` and `training` at their defaults for your first trials. **Optional challenge:** At the same width and precision, change just the seed or training length. Can you catch more PVCs for the same number of bytes? Use the same settings to repeat a trial. Scores can vary slightly across platforms.

Numbers in the table use the **practice recordings**. A trial can pass the practice floors and still miss a floor on the organizer’s unseen recordings. The plot’s dashed line marks the sensitivity floor. Point color shows whether the trial meets **both** practice floors. Run the next cell once to start your trial history.

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.ticker import PercentFormatter

# Reuse one trained model for both weight formats. Keep scores by trial number.
fitted = {(BASELINE_WIDTH, SEED, 'standard'): baseline_model}
trials = {}


def record_trial(key, payload, metrics):
    passes = metrics['accuracy'] >= accuracy_floor and metrics['sensitivity'] >= sensitivity_floor
    trials[key] = dict(metrics, number=len(trials) + 1, payload=payload, passes=passes)


def run_trial(width, bits, seed, training):
    if width not in WIDTHS or bits not in (8, 32):
        raise ValueError('Pick one of the listed widths and use 8 or 32 bits.')
    if seed not in TRAINING_SEEDS or training not in TRAINING_EPOCHS:
        raise ValueError('Pick a listed seed and either "standard" or "short" training.')
    key = (width, bits, seed, training)
    if key not in trials:
        fit_key = (width, seed, training)
        if fit_key not in fitted:
            fitted[fit_key] = train(x_train, y_train, width, seed=seed, training=training)
        payload = encode(fitted[fit_key], bits)
        record_trial(key, payload, score(payload, x_val, y_val))
    return trials[key]


def smallest_passing_trial():
    passing = (row for row in trials.values() if row['passes'])
    return min(passing, key=lambda row: (row['bytes'], -row['sensitivity'], -row['accuracy']), default=None)


def show_trials():
    best = smallest_passing_trial()
    print('Trial  Architecture  Weights  Training / seed  Bytes  Accuracy  PVC sensitivity  Practice floors')
    for (width, bits, seed, training), row in trials.items():
        marker = '*' if row is best else ' '
        architecture = f'{INPUTS}-{width}-1'
        precision = 'INT8' if bits == 8 else 'FP32'
        configuration = f'{training} / {seed}'
        status = 'Pass' if row['passes'] else 'Below floor'
        print(f"{marker}{row['number']:>3}   {architecture:<12}  {precision:<7}  {configuration:<15}"
              f" {row['bytes']:>5,}  {row['accuracy']:>8.2%}  {row['sensitivity']:>15.2%}  {status}")

    fig, ax = plt.subplots(figsize=(8, 3.7), layout='constrained')
    for passed, color, marker, label in ((True, '#157467', 'o', 'Meets both practice floors'),
                                         (False, '#aa4e26', 'x', 'Below a practice floor')):
        group = [row for row in trials.values() if row['passes'] == passed]
        if group:
            ax.scatter([row['bytes'] for row in group], [row['sensitivity'] for row in group],
                       color=color, marker=marker, s=44, label=label, zorder=3)
    ax.axhline(sensitivity_floor, color='#4b5563', linestyle='--',
               label=f'{sensitivity_floor:.0%} PVC sensitivity floor')
    if best is not None:
        ax.scatter(best['bytes'], best['sensitivity'], s=170, facecolors='none',
                   edgecolors='#102e48', linewidths=1.6, label=f"Smallest passing trial #{best['number']}", zorder=4)
        print(f"* Smallest model meeting both practice floors: trial #{best['number']} ({best['bytes']:,} bytes).")
    else:
        print('No trial has met both practice floors yet.')
    ax.set(xlabel='Exported model size (bytes)', ylabel='Practice PVC sensitivity', xlim=(0, None))
    ax.yaxis.set_major_formatter(PercentFormatter(1))
    ax.grid(axis='y', alpha=0.2)
    ax.legend(loc='best')
    plt.show()


record_trial((BASELINE_WIDTH, 32, SEED, 'standard'), baseline, baseline_score)
record_trial((BASELINE_WIDTH, 8, SEED, 'standard'), int8_baseline, int8_score)
print('Trials #1 and #2 are your FP32 and INT8 baseline models.')

In [ ]:
width = 16
bits = 8
seed = 18
training = 'standard'

run_trial(width, bits, seed, training)
show_trials()

## D · Submit — 1 minute

Pick a trial number from the table. Give yourself room above both floors if you can. Enter the number and your team name below, run the cell, and give the facilitator the downloaded `.mgf` file. In local Jupyter, pick up the file from the notebook folder. Send one file per team.

The facilitator scores the file on separate recordings using the final floors shown in Stage A. The smallest qualifying model wins. Higher PVC sensitivity breaks a size tie, then higher accuracy. An exact tie shares rank.

In [ ]:
import re

team_name = ''
trial_number = None  # Replace with a trial number from your table.


def export_submission(team, number):
    if not isinstance(team, str) or not re.fullmatch(r'[A-Za-z0-9][A-Za-z0-9 _-]{0,39}', team.strip()):
        raise ValueError('Use 1–40 letters, digits, spaces, underscores, or hyphens. Start with a letter or digit.')
    by_number = {row['number']: row for row in trials.values()}
    if type(number) is not int or number not in by_number:
        raise ValueError('Choose a whole trial number from the results table.')
    trial = by_number[number]
    decode(trial['payload'])  # Confirm the chosen file is a valid ESP32 model before writing it.
    path = Path(f'{team.strip()}.mgf')
    path.write_bytes(trial['payload'])
    print(f"Trial #{number} → {path.name} ({path.stat().st_size:,} bytes)")
    print(f"Practice: {trial['accuracy']:.2%} accuracy · {trial['sensitivity']:.2%} PVC sensitivity")
    return path


if team_name or trial_number is not None:
    submission = export_submission(team_name, trial_number)
    try:
        from google.colab import files
    except ImportError:
        print(submission.resolve())
    else:
        files.download(str(submission))
else:
    print('Enter your team name and a trial number, then rerun this cell.')

What saved the most bytes? Which model missed the fewest PVCs? How might your choice change on a different recording?

[MIT-BIH Arrhythmia Database](https://physionet.org/content/mitdb/1.0.0/) · [WFDB beat annotation codes](https://physionet.org/physiotools/wpg/wpg_36.htm)